# Notebook 3: Memory Management

In this notebook, you will learn:

- The CUDA memory model (host vs device memory)
- Explicit memory management: `cudaMalloc`, `cudaMemcpy`, `cudaFree`
- Unified Memory: `cudaMallocManaged`
- Memory transfer patterns and their performance implications
- Pinned (page-locked) host memory

In [ ]:
%load_ext nvcc4jupyter

## 3.1 The CUDA Memory Model

The CPU and GPU have **separate memory spaces**. Data must be explicitly copied between them.

```
┌─────────────────┐          ┌─────────────────┐
│      CPU        │          │      GPU         │
│                 │          │                  │
│  Host Memory    │ ◄──────► │  Device Memory   │
│  (RAM)          │  PCIe    │  (VRAM)          │
│                 │  bus     │                  │
│  malloc()       │          │  cudaMalloc()    │
│  free()         │          │  cudaFree()      │
└─────────────────┘          └─────────────────┘
```

**Key insight:** Memory transfers over the PCIe bus are slow compared to GPU computation. Minimizing transfers is critical for performance.

| Memory | Access speed | Accessible from |
|--------|-------------|----------------|
| Host (RAM) | Fast for CPU | CPU only |
| Device (VRAM) | Fast for GPU | GPU only |
| PCIe transfer | ~12-32 GB/s | Bridge between the two |

## 3.2 Explicit Memory Management

The fundamental API:

```cpp
cudaMalloc(&d_ptr, size_in_bytes);                       // Allocate on GPU
cudaMemcpy(dst, src, size_in_bytes, direction);          // Copy data
cudaFree(d_ptr);                                         // Free GPU memory
```

Direction flags:
- `cudaMemcpyHostToDevice` (CPU → GPU)
- `cudaMemcpyDeviceToHost` (GPU → CPU)
- `cudaMemcpyDeviceToDevice` (GPU → GPU)

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

// Simple kernel: multiply each element by 2
__global__ void scale(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        data[idx] *= 2.0f;
    }
}

int main() {
    const int N = 8;
    const size_t bytes = N * sizeof(float);

    // Step 1: Allocate host memory and initialize
    float* h_data = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) {
        h_data[i] = (float)i;  // [0, 1, 2, 3, 4, 5, 6, 7]
    }

    printf("Before: ");
    for (int i = 0; i < N; i++) printf("%.0f ", h_data[i]);
    printf("\n");

    // Step 2: Allocate device memory
    float* d_data;
    cudaMalloc(&d_data, bytes);

    // Step 3: Copy data from host to device
    cudaMemcpy(d_data, h_data, bytes, cudaMemcpyHostToDevice);

    // Step 4: Launch kernel
    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;
    scale<<<grid_size, block_size>>>(d_data, N);

    // Step 5: Copy results back to host
    cudaMemcpy(h_data, d_data, bytes, cudaMemcpyDeviceToHost);

    printf("After:  ");
    for (int i = 0; i < N; i++) printf("%.0f ", h_data[i]);
    printf("\n");

    // Step 6: Free memory
    cudaFree(d_data);
    free(h_data);

    return 0;
}

### The Standard Pattern

```cpp
// 1. Allocate host + device
float* h_data = (float*)malloc(bytes);
float* d_data;
cudaMalloc(&d_data, bytes);

// 2. Initialize host data
// ... fill h_data ...

// 3. Copy to device
cudaMemcpy(d_data, h_data, bytes, cudaMemcpyHostToDevice);

// 4. Run kernel
kernel<<<grid, block>>>(d_data, N);

// 5. Copy back
cudaMemcpy(h_data, d_data, bytes, cudaMemcpyDeviceToHost);

// 6. Cleanup
cudaFree(d_data);
free(h_data);
```

You'll write this pattern hundreds of times. It becomes muscle memory.

## 3.3 Working with Multiple Arrays

Most kernels take multiple input/output arrays. Each needs its own allocation and transfer.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

// c[i] = a[i] + b[i]
__global__ void add_arrays(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        c[idx] = a[idx] + b[idx];
    }
}

int main() {
    const int N = 5;
    const size_t bytes = N * sizeof(float);

    // Host arrays
    float h_a[] = {1, 2, 3, 4, 5};
    float h_b[] = {10, 20, 30, 40, 50};
    float h_c[5];

    // Device arrays
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy inputs to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Launch
    add_arrays<<<1, 256>>>(d_a, d_b, d_c, N);

    // Copy output back
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Verify
    printf("a + b = c:\n");
    for (int i = 0; i < N; i++) {
        printf("  %.0f + %.0f = %.0f\n", h_a[i], h_b[i], h_c[i]);
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}

## 3.4 `cudaMemset` — Initialize Device Memory

You can zero-initialize device memory without transferring from host:

In [ ]:
%%cuda
#include <cstdio>

int main() {
    const int N = 8;
    float* d_data;
    cudaMalloc(&d_data, N * sizeof(float));

    // Set all bytes to 0 (works for zeroing floats/ints)
    cudaMemset(d_data, 0, N * sizeof(float));

    // Verify
    float h_data[8];
    cudaMemcpy(h_data, d_data, N * sizeof(float), cudaMemcpyDeviceToHost);

    printf("After cudaMemset(0): ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_data[i]);
    printf("\n");

    cudaFree(d_data);
    return 0;
}

**Note:** `cudaMemset` sets **bytes**, not values. Setting to 0 works for ints and floats (all-zero bits = 0.0f). Setting to any other value fills each byte with that value, which usually isn't what you want for floats. Use a kernel to set arbitrary values.

## 3.5 Unified Memory

Tired of manually copying data back and forth? **Unified Memory** creates a single pointer accessible from both CPU and GPU. The CUDA runtime handles transfers automatically.

```cpp
float* data;
cudaMallocManaged(&data, bytes);  // Accessible from CPU and GPU!
```

In [ ]:
%%cuda
#include <cstdio>

__global__ void scale(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        data[idx] *= 3.0f;
    }
}

int main() {
    const int N = 8;

    // Allocate Unified Memory — accessible from CPU and GPU
    float* data;
    cudaMallocManaged(&data, N * sizeof(float));

    // Initialize on CPU (no cudaMemcpy needed!)
    for (int i = 0; i < N; i++) {
        data[i] = (float)(i + 1);
    }

    printf("Before: ");
    for (int i = 0; i < N; i++) printf("%.0f ", data[i]);
    printf("\n");

    // Run on GPU (no cudaMemcpy needed!)
    scale<<<1, 256>>>(data, N);
    cudaDeviceSynchronize();  // MUST sync before CPU accesses data

    // Read on CPU (no cudaMemcpy needed!)
    printf("After:  ");
    for (int i = 0; i < N; i++) printf("%.0f ", data[i]);
    printf("\n");

    cudaFree(data);  // Works for managed memory too
    return 0;
}

### Unified Memory: Pros and Cons

**Pros:**
- Much simpler code — no manual copies
- Great for prototyping and learning
- Handles complex data structures (linked lists, trees) that are hard to copy manually

**Cons:**
- Can be slower than explicit copies (page fault overhead)
- Less control over when transfers happen
- Performance depends on access patterns and GPU architecture

**Rule of thumb:** Use Unified Memory for prototyping, explicit copies for production code where performance matters.

## 3.6 Pinned (Page-Locked) Host Memory

Normal `malloc` memory can be swapped to disk by the OS. **Pinned memory** is locked in physical RAM, enabling faster DMA transfers to the GPU.

```cpp
float* h_data;
cudaMallocHost(&h_data, bytes);  // Pinned host memory
// ... use it ...
cudaFreeHost(h_data);            // Free pinned memory
```

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <ctime>

int main() {
    const int N = 10000000;  // 10M elements
    const size_t bytes = N * sizeof(float);

    // Regular host memory
    float* h_regular = (float*)malloc(bytes);

    // Pinned host memory
    float* h_pinned;
    cudaMallocHost(&h_pinned, bytes);

    // Device memory
    float* d_data;
    cudaMalloc(&d_data, bytes);

    // Initialize both
    for (int i = 0; i < N; i++) {
        h_regular[i] = 1.0f;
        h_pinned[i] = 1.0f;
    }

    // Time regular transfer
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    cudaMemcpy(d_data, h_regular, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float ms_regular = 0;
    cudaEventElapsedTime(&ms_regular, start, stop);

    // Time pinned transfer
    cudaEventRecord(start);
    cudaMemcpy(d_data, h_pinned, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float ms_pinned = 0;
    cudaEventElapsedTime(&ms_pinned, start, stop);

    printf("Transfer of %.1f MB:\n", bytes / 1e6);
    printf("  Regular memory: %.2f ms (%.2f GB/s)\n",
           ms_regular, bytes / ms_regular / 1e6);
    printf("  Pinned memory:  %.2f ms (%.2f GB/s)\n",
           ms_pinned, bytes / ms_pinned / 1e6);
    printf("  Speedup: %.1fx\n", ms_regular / ms_pinned);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_data);
    cudaFreeHost(h_pinned);
    free(h_regular);
    return 0;
}

### Pinned memory guidelines:
- **~2x faster** transfers than regular memory (avoids OS paging)
- **Don't overuse** — pinned memory reduces RAM available to the OS
- Required for **asynchronous** transfers (overlapping compute and transfer)
- Use for large buffers that are frequently transferred

## 3.7 Memory Types Summary

| Memory Type | Allocation | Accessible From | Speed | Use Case |
|-------------|-----------|----------------|-------|----------|
| Host (pageable) | `malloc` | CPU | - | General CPU data |
| Host (pinned) | `cudaMallocHost` | CPU | - | Fast transfers |
| Device (global) | `cudaMalloc` | GPU | ~900 GB/s | Main GPU storage |
| Unified | `cudaMallocManaged` | Both | Auto-managed | Prototyping |
| Shared | `__shared__` | Block's threads | ~12 TB/s | Fast scratch pad (Notebook 8+) |
| Registers | Automatic | Single thread | Fastest | Local variables |

## 3.8 Common Mistakes

### 1. Accessing device memory from host (segfault!)
```cpp
float* d_data;
cudaMalloc(&d_data, 100 * sizeof(float));
d_data[0] = 1.0f;  // CRASH — d_data is a GPU pointer!
```

### 2. Forgetting to sync before reading results
```cpp
kernel<<<grid, block>>>(d_data, N);
// cudaDeviceSynchronize();  // Oops, forgot this!
cudaMemcpy(h_data, d_data, bytes, cudaMemcpyDeviceToHost);
// Actually, cudaMemcpy is synchronous — it implicitly syncs.
// But with Unified Memory, you MUST call cudaDeviceSynchronize.
```

### 3. Wrong transfer direction
```cpp
// Wanted to copy host → device, but used wrong flag:
cudaMemcpy(d_data, h_data, bytes, cudaMemcpyDeviceToHost);  // WRONG direction!
```

### 4. Memory leak — forgetting `cudaFree`
```cpp
for (int i = 0; i < 1000; i++) {
    float* d_temp;
    cudaMalloc(&d_temp, bytes);  // Leaked 999 times!
}
```

## 3.9 Key Takeaways

1. CPU and GPU have **separate memory** — data must be explicitly transferred
2. **The pattern:** `cudaMalloc → cudaMemcpy(H→D) → kernel → cudaMemcpy(D→H) → cudaFree`
3. **Unified Memory** (`cudaMallocManaged`) simplifies code but may sacrifice performance
4. **Pinned memory** (`cudaMallocHost`) gives ~2x faster transfers
5. **Minimize transfers** — the PCIe bus is the bottleneck in many programs
6. Always match your `cudaMalloc` with `cudaFree` (no leaks!)

## Exercises

**Exercise 3.1:** Allocate an array of 1 million integers on the GPU. Fill it on the CPU with values `0, 1, 2, ..., 999999`, transfer to GPU, run a kernel that squares each value, transfer back, and verify `data[42] == 42*42`.

**Exercise 3.2:** Rewrite Exercise 3.1 using Unified Memory. Compare the code complexity.

**Exercise 3.3:** Write a program with two arrays `a` and `b` where a kernel computes `a[i] = a[i] + b[i]` in-place (modifying `a`). How many transfers are needed vs a version with a separate output array `c`?

In [ ]:
%%cuda
// Exercise 3.1 — Explicit memory management
#include <cstdio>
#include <cstdlib>

__global__ void square_kernel(int* data, int N) {
    // TODO: Square each element
}

int main() {
    const int N = 1000000;
    // TODO: Allocate, initialize, transfer, compute, verify
    return 0;
}

In [ ]:
%%cuda
// Exercise 3.2 — Same problem with Unified Memory
#include <cstdio>

__global__ void square_kernel(int* data, int N) {
    // TODO
}

int main() {
    const int N = 1000000;
    // TODO: Use cudaMallocManaged
    return 0;
}

---
**Next:** [Notebook 4 — Vector Addition](04_Vector_Addition.ipynb) — Your first complete, practical CUDA program.